In [1]:
#Imports
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

In [ ]:
Raw_Path = "FREDMD-Current.csv"
series = ["RPI", "UNRATE", "CPIAUCSL", "GS5", "DPCERA3M086SBEA"]

raw_csv = pd.read_csv(Raw_Path)
transform_codes = raw_csv.iloc[0]
data = raw_csv.iloc[1:].reset_index(drop=True)
data["sasdate"] = pd.to_datetime(data["sasdate"])
data = data.set_index("sasdate")
data = data.apply(pd.to_numeric, errors="coerce")
df = data[series].copy()

In [3]:
def transform(series, code):
    if code == 1:
        return series
    elif code == 2:
        return series.diff()
    elif code == 3:
        return series .diff().diff()
    elif code == 4:
        return np.log(series)
    elif code == 5:
        return np.log(series).diff()
    elif code == 6:
        return np.log(series).diff().diff()
    elif code == 7:
        return (series / series.shift(1) - 1).diff()
    else:
        return series

df = df.interpolate(method = "linear")

series_tracker = {}
for col in series:
    s = transform(df[col], transform_codes[col]).dropna()
    series_tracker[col] = s

In [4]:

def make_windows_volnorm(arr, p, vol_window=60):
    X, Y, vols = [], [], []
    for i in range(p, len(arr)):
        start = max(0, i - vol_window)
        vol = arr[start:i].std() + 1e-5      
        X.append(arr[i-p:i] / vol)           
        Y.append(arr[i] / vol)             
        vols.append(vol)                     
    X = np.array(X, dtype=np.float32)[..., None]  
    Y = np.array(Y, dtype=np.float32)[:, None]     
    vols = np.array(vols, dtype=np.float32)[:, None]
    return torch.from_numpy(X), torch.from_numpy(Y), torch.from_numpy(vols)

def make_windows_plain(arr, p):
    X, Y = [], []
    for i in range(p, len(arr)):
        X.append(arr[i-p:i])
        Y.append(arr[i])
    X = np.array(X, dtype=np.float32)[..., None]
    Y = np.array(Y, dtype=np.float32)[:, None]
    return torch.from_numpy(X), torch.from_numpy(Y)

def time_split_vn(X, Y, vols, train=0.70, val=0.10):
    n = len(X)
    tr = int(n * train); va = int(n * (train + val))
    return (X[:tr], Y[:tr], vols[:tr],
            X[tr:va], Y[tr:va], vols[tr:va],
            X[va:], Y[va:], vols[va:])

In [ ]:
loss_function = nn.MSELoss()

def ar1(X_tr, Y_tr, X_te, Y_te):
    x_tr = X_tr[:, -1, 0].numpy(); y_tr = Y_tr[:, 0].numpy()
    A = np.vstack([x_tr, np.ones(len(x_tr))]).T
    a, b = np.linalg.lstsq(A, y_tr, rcond=None)[0]
    x_te = X_te[:, -1, 0].numpy()
    pred = a * x_te + b
    return float(np.mean((pred - Y_te[:, 0].numpy()) ** 2))

def train_model(model, X_tr, Y_tr, X_val, Y_val, num_epochs = 500, learning_rate = 0.01, patience = 30, verbose = False):
    optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

    best_val = float('inf')
    best_state = None
    improve = 0

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        loss = loss_function(model(X_tr), Y_tr)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_function(model(X_val), Y_val)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            improve = 0
        else:
            improve += 1

        if verbose and epoch % 30 == 0:
            print(f'Epoch {epoch}, Training Loss: {loss.item()}, Validation Loss: {val_loss.item()}')

        if improve >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val

def evaluate_vn(model, X, Y, vols):
    with torch.no_grad():
        pred = model(X) * vols
        actual = Y * vols
        return loss_function(pred, actual).item()

In [ ]:
# FCNN
class FCNN(nn.Module):
    def __init__(self, input_size=1, p=12, hidden_size=50, output_size=1):
        super().__init__()
        self.hidden = nn.Linear(p, hidden_size)
        self.activation = nn.ReLU()
        self.output = nn.Linear(hidden_size, output_size)
    def forward(self, x):        
        x = x.squeeze(-1)            
        h = self.activation(self.hidden(x))
        return self.output(h)

In [ ]:
# Run FCNN
lags = 12
seeds = [0, 1, 2, 3, 4]
vn_results_fcnn = {}

for name in series:
    arr = series_tracker[name].values.astype(np.float32)
        # vol-norm windows for the model
    X, Y, vols = make_windows_volnorm(arr, lags)
    (X_tr, Y_tr, v_tr, X_val, Y_val, v_val, X_te, Y_te, v_te) = time_split_vn(X, Y, vols)

    Xp, Yp = make_windows_plain(arr, lags)
    n = len(Xp); trp = int(n*0.70); vap = int(n*0.80)
    base_mse = ar1(Xp[:trp], Yp[:trp], Xp[vap:], Yp[vap:])

    errs = []
    for seed in seeds:
        set_seed(seed)
        model = FCNN(input_size=1, p=lags, hidden_size=50)
        model, _ = train_model(model, X_tr, Y_tr, X_val, Y_val)
        errs.append(evaluate_vn(model, X_te, Y_te, v_te))
    errs = np.array(errs)
    vn_results_fcnn[name] = {"baseline": base_mse, "mean": errs.mean(), "std": errs.std()}
    print(f"{name:18s} AR(1)={base_mse:.6f}  FCNN(volnorm)={errs.mean():.6f} +/- {errs.std():.6f}")

table = pd.DataFrame(vn_results_fcnn).T
print("\n", table)

RPI                AR(1)=0.000471  FCNN(volnorm)=0.000566 +/- 0.000030
UNRATE             AR(1)=0.781290  FCNN(volnorm)=1.036994 +/- 0.047067
CPIAUCSL           AR(1)=0.000007  FCNN(volnorm)=0.000006 +/- 0.000000
GS5                AR(1)=0.034780  FCNN(volnorm)=0.036508 +/- 0.000696
DPCERA3M086SBEA    AR(1)=0.000209  FCNN(volnorm)=0.000200 +/- 0.000006

                  baseline      mean           std
RPI              0.000471  0.000566  3.012968e-05
UNRATE           0.781290  1.036994  4.706729e-02
CPIAUCSL         0.000007  0.000006  1.644072e-07
GS5              0.034780  0.036508  6.961385e-04
DPCERA3M086SBEA  0.000209  0.000200  6.141950e-06
